# Reconocimiento de Actividad Humana con Machine Learning
### Proyecto Final — Machine Learning

---

| | |
|---|---|
| **Equipo** | Grupo XX |
| **Integrantes** | [Nombre 1] · [Nombre 2] · [Nombre 3] |
| **Fecha** | [Fecha de entrega] |
| **Dataset** | UCI HAR — Human Activity Recognition Using Smartphones |

---

> **Objetivo de la Fase 3:** Comunicar los resultados del proyecto de forma clara y coherente, integrando el trabajo de las tres fases con una narrativa que explique decisiones, limitaciones y aprendizajes.

## Configuración del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request, zipfile, time, warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Entorno listo.')

In [ ]:
# Carga de datos
DATA_PATH = 'UCI HAR Dataset'
if not os.path.exists(DATA_PATH):
    URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
    urllib.request.urlretrieve(URL, 'har_dataset.zip')
    with zipfile.ZipFile('har_dataset.zip', 'r') as z:
        z.extractall('.')
    os.remove('har_dataset.zip')

features = pd.read_csv(f'{DATA_PATH}/features.txt', sep=r'\s+', header=None, names=['idx','feature'])
feature_names = features['feature'].tolist()
X_train = pd.read_csv(f'{DATA_PATH}/train/X_train.txt', sep=r'\s+', header=None, names=feature_names)
y_train = pd.read_csv(f'{DATA_PATH}/train/y_train.txt', sep=r'\s+', header=None, names=['Activity']).squeeze()
X_test  = pd.read_csv(f'{DATA_PATH}/test/X_test.txt',  sep=r'\s+', header=None, names=feature_names)
y_test  = pd.read_csv(f'{DATA_PATH}/test/y_test.txt',  sep=r'\s+', header=None, names=['Activity']).squeeze()

ACTIVITY_LABELS = {1:'WALKING',2:'WALKING_UPSTAIRS',3:'WALKING_DOWNSTAIRS',4:'SITTING',5:'STANDING',6:'LAYING'}
y_train_labels = y_train.map(ACTIVITY_LABELS)
y_test_labels  = y_test.map(ACTIVITY_LABELS)
print('Datos cargados:', X_train.shape, X_test.shape)

---
## 1. Introducción

### ¿Por qué es relevante el reconocimiento de actividad humana (HAR)?

El reconocimiento automático de actividades físicas a partir de sensores inerciales tiene aplicaciones directas en salud digital, rehabilitación, deporte y dispositivos móviles. Poder determinar si una persona está caminando, sentada o acostada —sin intervención manual— permite construir sistemas de monitoreo continuo que son tanto no invasivos como de bajo costo.

En este proyecto trabajamos con el dataset **UCI HAR**, que contiene mediciones de acelerómetro y giroscopio de 30 voluntarios (edades 19–48) realizando 6 actividades cotidianas con un smartphone en la cintura. A partir de las señales se extrajeron **561 features** en dominios de tiempo y frecuencia, lo que hace de este problema un caso real de clasificación multiclase en alta dimensionalidad.

In [ ]:
# Descripción del dataset
print('Dataset UCI HAR')
print(f'  Sujetos:           30 voluntarios')
print(f'  Actividades:       {len(ACTIVITY_LABELS)} clases')
print(f'  Features:          {X_train.shape[1]}')
print(f'  Muestras train:    {X_train.shape[0]}')
print(f'  Muestras test:     {X_test.shape[0]}')
print(f'  Clases:            {list(ACTIVITY_LABELS.values())}')

---
## 2. Resumen Fase 1: Hallazgos del EDA

*[Redactar aquí un párrafo narrativo con los principales hallazgos del análisis exploratorio: balance de clases, features más discriminativas, observaciones del PCA, etc. No copiar todas las visualizaciones — elegir las más relevantes.]*

In [ ]:
# --- Visualización 1 (de Fase 1): Distribución de clases ---
class_counts = y_train_labels.value_counts()
total = len(y_train_labels)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(class_counts.index, class_counts.values,
              color=sns.color_palette('tab10', n_colors=6))
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{count}\n({count/total:.1%})', ha='center', va='bottom', fontsize=9)
ax.set_title('Distribución de Actividades — Set de Entrenamiento', fontsize=13)
ax.set_xlabel('Actividad')
ax.set_ylabel('Número de muestras')
plt.tight_layout()
plt.show()

In [ ]:
# --- Visualización 2 (de Fase 1): PCA ---
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_train)
df_pca = pd.DataFrame(X_pca, columns=['PC1','PC2'])
df_pca['Activity'] = y_train_labels.values

fig, ax = plt.subplots(figsize=(9, 6))
palette = sns.color_palette('tab10', n_colors=6)
for i, act in enumerate(ACTIVITY_LABELS.values()):
    mask = df_pca['Activity'] == act
    ax.scatter(df_pca.loc[mask,'PC1'], df_pca.loc[mask,'PC2'],
               label=act, alpha=0.4, s=10, color=palette[i])
ax.set_title(f'PCA (var. explicada: {sum(pca.explained_variance_ratio_):.2%})', fontsize=13)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(title='Actividad', bbox_to_anchor=(1.05,1), loc='upper left', markerscale=2)
plt.tight_layout()
plt.show()

**Principales hallazgos del EDA:**

*[Describe los hallazgos más importantes: ¿el dataset estaba balanceado? ¿Qué features separaban mejor las clases? ¿Qué muestra el PCA sobre la separabilidad? Mínimo 3 observaciones con justificación.]*

---
## 3. Resumen Fase 2: Benchmarking de Modelos

Entrenamos 6 modelos (incluyendo baseline trivial) con parámetros por defecto de sklearn y los comparamos en el set de prueba usando F1-macro como métrica principal.

In [ ]:
# Re-entrenar todos los modelos para el notebook integrado
import time

def entrenar(nombre, modelo):
    t0 = time.time()
    modelo.fit(X_train, y_train)
    t = time.time() - t0
    y_pred = modelo.predict(X_test)
    return {
        'Modelo': nombre,
        'Accuracy_Test': accuracy_score(y_test, y_pred),
        'F1_Macro':   f1_score(y_test, y_pred, average='macro'),
        'F1_Weighted':f1_score(y_test, y_pred, average='weighted'),
        'Tiempo_Entrenamiento': round(t, 3),
        'modelo_obj': modelo,
        'y_pred': y_pred
    }

modelos = [
    ('DummyClassifier',    DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)),
    ('Logistic Regression',LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)),
    ('KNN (k=5)',          KNeighborsClassifier(n_neighbors=5)),
    ('Decision Tree',      DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ('Random Forest',      RandomForestClassifier(random_state=RANDOM_STATE)),
    ('SVM (RBF)',          SVC(kernel='rbf', random_state=RANDOM_STATE)),
]

resultados = [entrenar(n, m) for n, m in modelos]
print('Entrenamiento completado.')

In [ ]:
# Tabla de benchmarking
df_bench = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('modelo_obj','y_pred')}
    for r in resultados
]).sort_values('F1_Macro', ascending=False).reset_index(drop=True)

print(df_bench.to_string(index=False))

In [ ]:
# Barplot comparativo
fig, ax = plt.subplots(figsize=(10, 5))
palette = ['#d62728' if r['Modelo']=='DummyClassifier' else '#1f77b4' for r in resultados]
df_plot = df_bench.copy()
bars = ax.barh(df_plot['Modelo'], df_plot['F1_Macro'],
               color=['#2ca02c' if i==0 else '#1f77b4' if j!='DummyClassifier' else '#d62728'
                      for i,(j) in enumerate(df_plot['Modelo'])])
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_xlim(0, 1.08)
ax.set_xlabel('F1-Score Macro')
ax.set_title('Benchmarking de Modelos — F1-Score Macro (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

**Modelo seleccionado:** *[Nombre del modelo ganador]*

**Justificación:** *[¿Por qué este modelo? Considera F1-macro, tiempo de entrenamiento y cualquier otro factor relevante. ¿Existe un trade-off entre rendimiento y costo computacional que el equipo deba mencionar?]*

---
## 4. Análisis de Errores

¿Dónde falla el modelo seleccionado? Esta sección explica qué actividades se confunden y por qué.

In [ ]:
# Matriz de confusión del mejor modelo
mejor = resultados[df_bench.index[0]]
y_pred_best = mejor['y_pred']

fig, ax = plt.subplots(figsize=(9, 7))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(
        y_test_labels, pd.Series(y_pred_best).map(ACTIVITY_LABELS),
        labels=list(ACTIVITY_LABELS.values())
    ),
    display_labels=list(ACTIVITY_LABELS.values())
)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title(f'Matriz de Confusión — {mejor["Modelo"]}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# F1 por clase del mejor modelo
report = classification_report(
    y_test_labels,
    pd.Series(mejor['y_pred']).map(ACTIVITY_LABELS),
    output_dict=True
)
f1_por_clase = {cls: report[cls]['f1-score'] for cls in ACTIVITY_LABELS.values() if cls in report}
df_f1 = pd.Series(f1_por_clase, name='F1').sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if v < 0.85 else '#2ca02c' for v in df_f1.values]
df_f1.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0.85, linestyle='--', color='gray', alpha=0.7, label='Umbral 0.85')
ax.set_xlabel('F1-Score')
ax.set_title(f'F1 por Actividad — {mejor["Modelo"]}', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Actividades difíciles:**

*[¿Qué actividades tienen el F1 más bajo? ¿Con cuáles otras se confunden (observar la matriz de confusión)? ¿Tiene sentido físico esta confusión? Por ejemplo: SITTING y STANDING producen señales de acelerómetro similares porque ambas son actividades estáticas.]*

---
## 5. Limitaciones del Modelo

*[Responder con desarrollo propio. Guías de reflexión:]*

- **Generalización a nuevos sujetos:** El modelo fue entrenado con 21 sujetos y evaluado en 9. ¿Qué pasaría con sujetos muy diferentes (adultos mayores, personas con movilidad reducida)?
- **Sesgo del dispositivo:** Los datos provienen de un único modelo de smartphone (Samsung Galaxy S II, 50 Hz). ¿Funcionaría igual en un reloj inteligente o un iPhone?
- **Features precomputadas:** Trabajamos con 561 features ya extraídas, no con señales crudas. El pipeline de extracción de features es una caja negra para nosotros.
- **Actividades limitadas:** Solo se modelan 6 actividades. El mundo real tiene decenas de actividades y transiciones entre ellas.
- **Mejoras posibles:** ¿Qué agregarían? (Optimización de hiperparámetros, deep learning sobre señales crudas, más actividades, validación por sujeto, etc.)

---
## 6. Conclusión

*[Escribir la conclusión del equipo. Debe responder:]*

1. ¿Cuál fue el modelo que mejor clasificó las actividades y por qué creen que fue superior?
2. ¿Cuáles fueron los tres aprendizajes más importantes del proyecto para el equipo?
3. ¿Qué harían diferente si tuvieran más tiempo o recursos?
4. ¿Qué implicancias prácticas tiene este tipo de modelo en aplicaciones reales de salud o deporte?

In [ ]:
print('=' * 55)
print('RESUMEN FINAL DEL PROYECTO')
print('=' * 55)
print(f'Dataset:           UCI HAR (10,299 muestras, 561 features)')
print(f'Modelos evaluados: {len(df_bench)}')
print(f'Mejor modelo:      {df_bench.iloc[0]["Modelo"]}')
print(f'F1-Macro (test):   {df_bench.iloc[0]["F1_Macro"]:.4f}')
print(f'Accuracy (test):   {df_bench.iloc[0]["Accuracy_Test"]:.4f}')
print('=' * 55)

---
---
## BONO: Interpretabilidad del Modelo

> ⭐ Esta sección es **opcional** y otorga hasta +1.0 punto sobre la nota final.

Entrenar un modelo y saber que predice bien no es suficiente en aplicaciones reales (especialmente en salud). Esta sección introduce técnicas para entender **por qué** el modelo toma sus decisiones.

### B.1 Importancia de Features (Random Forest)

In [ ]:
# TODO B1: Extraer feature importances del Random Forest entrenado
rf_model = next(r['modelo_obj'] for r in resultados if 'Random Forest' in r['Modelo'])

# TODO: Obtener importancias y asociarlas a los nombres de features
feature_importances = pd.Series(
    rf_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print('Top 10 features más importantes:')
print(feature_importances.head(10))

In [ ]:
# TODO B2: Graficar top-20 features más importantes
top20 = feature_importances.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
top20[::-1].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Importancia (Gini)')
ax.set_title('Top-20 Features más Importantes — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

**Análisis de importancia de features:**

*[¿Qué tipo de feature predomina en el top-20: dominio tiempo (t) o frecuencia (f)? ¿Acelerómetro (Acc) o giroscopio (Gyro)? ¿Cuerpo (Body) o gravedad (Gravity)? ¿Tiene sentido físico que esas features sean las más relevantes para distinguir actividades?]*

### B.2 Valores SHAP

In [ ]:
# TODO B3: Instalar e importar SHAP
# !pip install shap --quiet
import shap
shap.initjs()
print('SHAP importado correctamente.')

In [ ]:
# TODO B4: Crear explicador SHAP para el mejor modelo
# Para Random Forest usa TreeExplainer (rápido)
# Para otros modelos usa KernelExplainer con una muestra pequeña (200 obs)

# Opción A — Random Forest (recomendado por velocidad):
explainer = shap.TreeExplainer(rf_model)

# Opción B — Modelo genérico (descomentar si el mejor modelo no es tree-based):
# muestra = shap.sample(X_test, 200, random_state=RANDOM_STATE)
# explainer = shap.KernelExplainer(mejor['modelo_obj'].predict_proba, muestra)

print('Explicador SHAP creado.')

In [ ]:
# TODO B5: Calcular valores SHAP (usar muestra de 200 obs para velocidad)
muestra_test = X_test.sample(200, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(muestra_test)
print(f'Forma de shap_values: {np.array(shap_values).shape}')

In [ ]:
# TODO B6: SHAP Summary Plot — features más influyentes globalmente
shap.summary_plot(
    shap_values,
    muestra_test,
    class_names=list(ACTIVITY_LABELS.values()),
    max_display=20
)

In [ ]:
# TODO B7: SHAP Waterfall Plot — análisis de un caso específico
# Elegir una muestra del test set y explicar la decisión del modelo
idx_muestra = 0  # TODO: cambiar por un índice interesante

# Para modelos multiclase, elegir la clase de interés (ej. clase 0)
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0][idx_muestra],
        base_values=explainer.expected_value[0],
        data=muestra_test.iloc[idx_muestra],
        feature_names=feature_names
    )
)

**Análisis de interpretabilidad:**

Responder las siguientes preguntas:

1. **¿Las features más importantes tienen sentido físico para HAR?** *[¿Cuáles son y por qué creen que son relevantes para distinguir actividades cotidianas?]*

2. **¿El modelo usa features de tiempo, frecuencia, o ambas?** *[Analizar el tipo de features dominantes en el summary plot.]*

3. **¿Encontraron algún hallazgo sorprendente o contra-intuitivo?** *[Por ejemplo: una feature que no esperaban que fuera importante, o una actividad cuya explicación SHAP no tiene sentido inmediato.]*

4. **Si tuvieran que explicar este modelo a un médico o kinesiólogo, ¿cómo lo harían?** *[Usar el análisis SHAP para dar una explicación en lenguaje no técnico de cómo el modelo toma decisiones.]*